# Captcha Image Binarization and Renaming Preprocessing

### Functionality
1. Read `.txt` label files from the `data_label` directory.
2. Locate corresponding `.png` images in `data_collection` based on label filenames.
3. Use OpenCV to **binarize** the images and convert pixel values from `0/255` to a strict `0/1` matrix.
4. Save the processed data in `.npy` format using the **label content** as the filename.


In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt

## 1. Configure Path Parameters

In [ ]:
BASE_DIR = "absolute_path_to_project_directory"

# Input directories
IMG_DIR = os.path.join(BASE_DIR, "data_collection")
LABEL_DIR = os.path.join(BASE_DIR, "data_label")

# Output directory 
OUTPUT_DIR = "absolute_path_to_output_directory"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Image source directory: {IMG_DIR}")
print(f"Label source directory: {LABEL_DIR}")
print(f"Results storage directory: {OUTPUT_DIR}")

## 2. Define Processing Functions
Core Logic: Read Label -> Locate Image -> Binarize -> Convert to 0/1 -> Save

In [ ]:
def clean_filename(name):
    """Clean filename by removing illegal characters"""
    illegal_chars = ['/', '\\', ':', '*', '?', '"', '<', '>', '|', '\n', '\r']
    for char in illegal_chars:
        name = name.replace(char, '_')
    return name.strip()

def process_batch(img_dir, label_dir, output_dir):
    label_files = list(Path(label_dir).glob("*.txt"))
    
    if not label_files:
        print(" No .txt label files found!")
        return

    print(f" Starting processing of {len(label_files)} files...")
    
    success = 0
    failed = 0
    skipped = 0

    for txt_path in label_files:
        try:
            # 1. Read label content
            with open(txt_path, 'r', encoding='utf-8') as f:
                label_text = f.read().strip()
            
            if not label_text:
                skipped += 1
                continue

            # 2. Locate corresponding image
            img_name = txt_path.stem + ".png"
            img_path = Path(img_dir) / img_name

            if not img_path.exists():
                print(f"Image missing: {img_name} (linked to label: {txt_path.name})")
                failed += 1
                continue

            # 3. Read and Binarize (OpenCV)
            # IMREAD_GRAYSCALE: Load as grayscale (single channel)
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                failed += 1
                continue

            # Use Otsu's method for automatic thresholding
            # In 'binary', background is 0 and text is 255 (or vice versa depending on source)
            _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            # 4. Convert to strict 0 and 1 
            # Logic: If pixel > 127 set to 1, else 0
            matrix_01 = (binary > 127).astype(np.uint8)

            # 5. Build output filename
            safe_name = clean_filename(label_text)
            if not safe_name:
                safe_name = txt_path.stem 
            
            # Handle duplicates: append counter if filename already exists
            output_filename = f"{safe_name}.npy"
            output_path = Path(output_dir) / output_filename
            
            counter = 1
            while output_path.exists():
                output_filename = f"{safe_name}_{counter}.npy"
                output_path = Path(output_dir) / output_filename
                counter += 1

            # 6. Save as .npy file
            np.save(str(output_path), matrix_01)
            
            success += 1
            if success % 500 == 0:
                print(f"   ...Processed {success} images")

        except Exception as e:
            print(f"Error processing {txt_path.name}: {e}")
            failed += 1

    print("-" * 30)
    print(f"Success: {success}")
    print(f"Failed: {failed}")
    print(f"Skipped: {skipped}")
    print(f"Results saved to: {output_dir}")

# Execute processing
process_batch(IMG_DIR, LABEL_DIR, OUTPUT_DIR)

## 3. Verify Results 
Randomly select one processed `.npy` file to check its shape, data type, unique values (should be 0/1), and display a preview.

In [ ]:
import random

# Get all generated npy files
npy_files = list(Path(OUTPUT_DIR).glob("*.npy"))

if npy_files:
    # Random selection
    sample_file = random.choice(npy_files)
    data = np.load(str(sample_file))
    
    print(f"Sample file: {sample_file.name}")
    print(f"Shape (Height, Width): {data.shape}")
    print(f"Data Type: {data.dtype}")
    print(f"Unique Values (Should be 0 and 1): {np.unique(data)}")
    print(f"Min: {data.min()}, Max: {data.max()}")
    
    # Visualization
    plt.figure(figsize=(10, 4))
    plt.imshow(data, cmap='gray') # Display as grayscale
    plt.title(f"Preview: {sample_file.name}\nShape: {data.shape}, Values: {np.unique(data)}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No .npy files found. Please check if the previous step was successful.")

### Data Cleaning: Standardize naming and remove duplicates

In [ ]:
import os
from pathlib import Path
from collections import defaultdict

output_dir = "path_to_binarized_output_directory"
npy_files = list(Path(output_dir).glob("*.npy"))

print(f"Analyzing naming conflicts for {len(npy_files)} files...")

# 1. Create mapping: Uppercase Target Name -> [List of original file paths]
groups = defaultdict(list)
for file_path in npy_files:
    target_name = file_path.stem.upper() + ".npy"
    groups[target_name].append(file_path)

rename_count = 0
delete_count = 0
error_count = 0

print(f"Executing renaming and deduplication...")

for target_name, file_list in groups.items():
    target_path = Path(output_dir) / target_name
    
    if len(file_list) == 1:
        # Case A: Only one file exists, simply rename to uppercase
        file_path = file_list[0]
        if file_path.name != target_name:
            try:
                file_path.rename(target_path)
                rename_count += 1
            except Exception as e:
                print(f"Rename failed for {file_path.name}: {e}")
                error_count += 1
    else:
        # Case B: Multiple files map to the same uppercase name 
        # Strategy: Keep the first one, delete the rest
        # Prioritize keeping a file that is already uppercase for safety
        keep_file = None
        for fp in file_list:
            if fp.name == target_name:
                keep_file = fp
                break
        
        if keep_file is None:
            keep_file = file_list[0] # If none are uppercase, keep the first in the list
        
        # Delete all others
        for fp in file_list:
            if fp != keep_file:
                try:
                    fp.unlink()
                    delete_count += 1
                except Exception as e:
                    print(f"Delete failed for {fp.name}: {e}")
        
        # If the kept file isn't uppercase yet, rename it
        if keep_file.name != target_name:
            try:
                keep_file.rename(target_path)
                rename_count += 1
            except Exception as e:
                print(f"Failed to rename kept file {keep_file.name}: {e}")
                error_count += 1

print("-" * 30)
print(f"Processing Complete!")
print(f"Renamed: {rename_count}")
print(f"Duplicates Deleted: {delete_count}")
print(f"Errors: {error_count}")

# Final Verification
final_files = [f.name for f in Path(output_dir).glob("*.npy")]
lowercase_files = [f for f in final_files if any(c.islower() for c in f)]

print(f"\n Total files remaining: {len(final_files)}")
if lowercase_files:
    print(f"Warning: {len(lowercase_files)} files still contain lowercase letters!")
    print("First 10 examples:", lowercase_files[:10])
else:
    print("All files are now uppercase and unique.")